# 🚀 Auto-Tuner: Fine-Tune Llama 3 with Unsloth (Research Grade v2)

Fine-tunes Llama 3 8B using Unsloth with full research-grade improvements:
- ✅ System prompt injection (reduces hallucination)
- ✅ Corrected LoRA config (lora_alpha=2x rank)
- ✅ Cosine LR scheduler + tuned hyperparameters
- ✅ Train/eval split with best checkpoint saving
- ✅ Perplexity + ROUGE + CUP hallucination evaluation

**Requirements:**
- Free T4 GPU (Runtime → Change runtime type → T4 GPU)
- Dataset uploaded to Google Drive

**Time:** ~25-35 minutes for 200 examples

## 📋 Configuration

**IMPORTANT:** Update these values before running!

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================================================

# Dataset filename (must exist in Drive: Finetune_Jobs/datasets/)
DATASET_FILENAME = "dataset-20251115_132156.jsonl"  # <- CHANGE THIS

# Model name (will be saved to Drive: Finetune_Jobs/models/)
MODEL_NAME = "customer-support-bot-v1"  # <- CHANGE THIS

# System prompt - defines your model's identity and scope.
# CRITICAL: This must match exactly what you use in Chat_With_Model.ipynb
SYSTEM_PROMPT = (
    "You are a helpful, accurate, and concise AI assistant. "
    "Answer questions clearly based on what you know with confidence. "
    "If you are unsure about something, say so honestly rather than guessing."
)  # <- CHANGE THIS to describe your specific bot's purpose

# Training settings (research-grade defaults)
MAX_SEQ_LENGTH     = 2048   # Context window size
BATCH_SIZE         = 2      # Keep at 2 for T4 GPU memory
GRADIENT_ACCUM     = 4      # Effective batch = 2 * 4 = 8
LEARNING_RATE      = 1e-4   # Halved from original to reduce overfitting
NUM_EPOCHS         = 3      # Training epochs
WARMUP_RATIO       = 0.1    # 10% of steps used for warmup (scales with dataset size)
WEIGHT_DECAY       = 0.05   # Increased regularization
LORA_RANK          = 16     # LoRA rank
LORA_ALPHA         = 32     # FIXED: must be 2x rank for proper scaling
LORA_DROPOUT       = 0.05   # Small dropout prevents LoRA overfitting
EVAL_SPLIT         = 0.1    # 10% held out for evaluation
EVAL_STEPS         = 20     # Evaluate every N steps

print("✅ Configuration loaded")
print(f"   Dataset    : {DATASET_FILENAME}")
print(f"   Model name : {MODEL_NAME}")
print(f"   LR         : {LEARNING_RATE}")
print(f"   LoRA alpha : {LORA_ALPHA} (rank={LORA_RANK}, ratio={LORA_ALPHA/LORA_RANK}x)")
print(f"   Scheduler  : cosine with warmup_ratio={WARMUP_RATIO}")

## 🔗 Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT       = "/content/drive/MyDrive/Finetune_Jobs"
DATASET_PATH     = f"{DRIVE_ROOT}/datasets/{DATASET_FILENAME}"
MODEL_OUTPUT_DIR = f"{DRIVE_ROOT}/models/{MODEL_NAME}"

os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models",   exist_ok=True)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"❌ Dataset not found: {DATASET_PATH}\n"
        f"Please upload {DATASET_FILENAME} to Drive: Finetune_Jobs/datasets/"
    )

print(f"✅ Drive mounted")
print(f"✅ Dataset found  : {DATASET_PATH}")
print(f"✅ Model will save: {MODEL_OUTPUT_DIR}")

## 📦 Step 2: Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install rouge-score  # for evaluation

print("✅ All dependencies installed")

## 🤖 Step 3: Load Base Model (Llama 3 8B)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("✅ Base model loaded (Llama 3 8B 4-bit)")
print(f"   BF16 supported : {is_bfloat16_supported()}")
print(f"   Max seq length : {MAX_SEQ_LENGTH}")

## 🎛️ Step 4: Add LoRA Adapters

**Key fix:** `lora_alpha=32` (2x rank) gives proper gradient scaling. Original value of 16 (1x rank) was too weak, causing the base model weights to dominate and limiting adaptation.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                       = LORA_RANK,
    target_modules          = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha              = LORA_ALPHA,    # FIXED: 32 (was 16)
    lora_dropout            = LORA_DROPOUT,  # FIXED: 0.05 (was 0)
    bias                    = "none",
    use_gradient_checkpointing = "unsloth",
    random_state            = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print("✅ LoRA adapters added")
print(f"   Trainable params : {trainable:,} ({100*trainable/total:.2f}% of total)")
print(f"   lora_alpha / rank = {LORA_ALPHA}/{LORA_RANK} = {LORA_ALPHA/LORA_RANK}x  ✅")

## 📊 Step 5: Load and Preview Dataset

In [ ]:
from datasets import load_dataset
import json

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(f"✅ Dataset loaded: {len(raw_dataset)} examples")
print(f"\nFirst raw example:")
print(json.dumps(raw_dataset[0], indent=2))

print(f"\n📋 Preview of conversations:")
for i in range(min(3, len(raw_dataset))):
    messages    = raw_dataset[i]['messages']
    user_msg    = next((m['content'] for m in messages if m['role'] == 'user'),      '')
    asst_msg    = next((m['content'] for m in messages if m['role'] == 'assistant'), '')
    print(f"\n  Example {i+1}:")
    print(f"    User      : {user_msg[:80]}...")
    print(f"    Assistant : {asst_msg[:80]}...")

## 🔄 Step 6: Format Dataset + Train/Eval Split

**Key fixes applied here:**
1. System prompt injected into every example — gives the model a bounded identity and dramatically reduces hallucination
2. Dataset split into 90% train / 10% eval for proper overfitting detection

In [ ]:
def format_chat(example):
    """
    Format a training example using Llama 3 chat template.

    CRITICAL FIX: System prompt is injected as the first message in every
    example. Without this, the model has no identity or scope boundary,
    which is a primary driver of hallucination and off-topic responses.
    """
    original_messages = example['messages']

    # Extract user and assistant turns (ignore any existing system message
    # from the dataset to avoid conflicts)
    user_content = next(
        (m['content'] for m in original_messages if m['role'] == 'user'), ''
    )
    asst_content = next(
        (m['content'] for m in original_messages if m['role'] == 'assistant'), ''
    )

    # Build messages WITH system prompt first
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": asst_content},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}


# Apply formatting
formatted_dataset = raw_dataset.map(format_chat, batched=False)

# Train / eval split — CRITICAL for detecting overfitting
split        = formatted_dataset.train_test_split(test_size=EVAL_SPLIT, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f"✅ Dataset formatted and split")
print(f"   Train examples : {len(train_dataset)}")
print(f"   Eval examples  : {len(eval_dataset)}")
print(f"\n📋 Formatted example (first 600 chars):")
print(train_dataset[0]['text'][:600])
print("...")
print("\n✅ Confirm system prompt is present in the formatted text above ^")

## 🏋️ Step 7: Configure Trainer

**Key fixes applied here:**
- `lr=1e-4` (was 2e-4) — reduces overfitting on small datasets
- `cosine` scheduler (was linear) — smoother decay, better generalization
- `warmup_ratio=0.1` (was 5 fixed steps) — scales correctly with dataset size
- `weight_decay=0.05` (was 0.01) — stronger regularization
- `packing=True` — more efficient for short examples
- `load_best_model_at_end=True` — saves best checkpoint, not last

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,   # ADDED: eval set for overfitting detection
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = True,           # FIXED: was False — more efficient packing
    args = TrainingArguments(
        per_device_train_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps  = GRADIENT_ACCUM,
        num_train_epochs             = NUM_EPOCHS,
        learning_rate                = LEARNING_RATE,   # FIXED: 1e-4 (was 2e-4)
        warmup_ratio                 = WARMUP_RATIO,    # FIXED: ratio-based (was fixed 5 steps)
        weight_decay                 = WEIGHT_DECAY,    # FIXED: 0.05 (was 0.01)
        lr_scheduler_type            = "cosine",        # FIXED: cosine (was linear)
        optim                        = "adamw_8bit",
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 1,
        seed                         = 3407,
        output_dir                   = "outputs",
        report_to                    = "none",

        # ADDED: Evaluation + best checkpoint saving
        evaluation_strategy          = "steps",
        eval_steps                   = EVAL_STEPS,
        save_strategy                = "steps",
        save_steps                   = EVAL_STEPS,
        load_best_model_at_end       = True,            # Saves best, not last checkpoint
        metric_for_best_model        = "eval_loss",
        greater_is_better            = False,
        save_total_limit             = 2,               # Keep only 2 checkpoints to save disk
    ),
)

effective_batch = BATCH_SIZE * GRADIENT_ACCUM
total_steps     = (len(train_dataset) // effective_batch) * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print("✅ Trainer configured")
print(f"   Effective batch size : {effective_batch}")
print(f"   Estimated total steps: {total_steps}")
print(f"   Warmup steps         : {warmup_steps} ({WARMUP_RATIO*100:.0f}% of steps)")
print(f"   Eval every           : {EVAL_STEPS} steps")

## 📈 Step 8: Measure Baseline Perplexity (Before Training)

Record perplexity of the base model BEFORE fine-tuning.
This is your research baseline. Compare with post-training perplexity in Step 11.

In [ ]:
import math
import torch

def compute_perplexity(model, tokenizer, texts, device="cuda", max_samples=20):
    """
    Compute perplexity on a list of texts.
    Lower perplexity = model fits the target distribution better.
    Report BEFORE and AFTER training for your research paper.
    """
    model.eval()
    total_loss   = 0.0
    total_tokens = 0
    samples      = texts[:max_samples]

    with torch.no_grad():
        for text in samples:
            inputs = tokenizer(
                text,
                return_tensors  = "pt",
                truncation      = True,
                max_length      = 512
            ).to(device)

            outputs      = model(**inputs, labels=inputs["input_ids"])
            n_tokens     = inputs["input_ids"].shape[1]
            total_loss   += outputs.loss.item() * n_tokens
            total_tokens += n_tokens

    avg_loss   = total_loss / max(total_tokens, 1)
    perplexity = math.exp(avg_loss)
    return round(perplexity, 4)


eval_texts = [row['text'] for row in eval_dataset.select(
    range(min(20, len(eval_dataset)))
)]

baseline_ppl = compute_perplexity(model, tokenizer, eval_texts)
print(f"📊 Baseline perplexity (before fine-tuning): {baseline_ppl}")
print(f"   (Record this for your research paper — compare with Step 11)")

## 🚀 Step 9: Start Training!

**Watch the `eval_loss` column.** If eval_loss starts rising while train_loss falls — that is overfitting. Training will automatically save the best checkpoint.

In [ ]:
import time

print("🚀 Starting training...")
print("   Monitor eval_loss column — should decrease alongside train_loss")
print("   If eval_loss rises while train_loss falls = overfitting detected\n")

start_time   = time.time()
trainer_stats = trainer.train()
elapsed      = time.time() - start_time

print(f"\n✅ Training complete!")
print(f"   Time         : {elapsed/60:.1f} minutes")
print(f"   Final train loss : {trainer_stats.training_loss:.4f}")

## 💾 Step 10: Save Best Model to Google Drive

In [ ]:
import json

# Save the best checkpoint (load_best_model_at_end=True loaded it automatically)
model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

# Save system prompt alongside model — must be reused at inference
config = {
    "model_name"             : MODEL_NAME,
    "dataset"                : DATASET_FILENAME,
    "base_model"             : "unsloth/llama-3-8b-bnb-4bit",
    "system_prompt"          : SYSTEM_PROMPT,
    "training_loss"          : float(trainer_stats.training_loss),
    "num_train_examples"     : len(train_dataset),
    "num_eval_examples"      : len(eval_dataset),
    "num_epochs"             : NUM_EPOCHS,
    "learning_rate"          : LEARNING_RATE,
    "lora_rank"              : LORA_RANK,
    "lora_alpha"             : LORA_ALPHA,
    "lora_dropout"           : LORA_DROPOUT,
    "training_time_minutes"  : round(elapsed / 60, 2),
    "timestamp"              : time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(f"{MODEL_OUTPUT_DIR}/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Model saved to: {MODEL_OUTPUT_DIR}")
print(f"✅ training_config.json saved (includes system prompt for inference)")
print(f"\n{json.dumps(config, indent=2)}")

## 📊 Step 11: Post-Training Evaluation

Compute perplexity and ROUGE scores after training. Compare perplexity to the baseline recorded in Step 8.

In [ ]:
from rouge_score import rouge_scorer as rouge_lib

# --- Perplexity after training ---
FastLanguageModel.for_inference(model)

post_ppl = compute_perplexity(model, tokenizer, eval_texts)

print("📊 Perplexity comparison:")
print(f"   Before fine-tuning : {baseline_ppl}")
print(f"   After  fine-tuning : {post_ppl}")
delta = baseline_ppl - post_ppl
pct   = 100 * delta / max(baseline_ppl, 1)
if delta > 0:
    print(f"   Improvement        : ↓ {delta:.2f} ({pct:.1f}% reduction) ✅")
else:
    print(f"   Change             : ↑ {abs(delta):.2f} (perplexity increased — check for overfitting)")


# --- ROUGE evaluation ---
def evaluate_rouge(model, tokenizer, eval_ds, max_samples=30, device="cuda"):
    """
    Generate predictions on eval set and compute ROUGE-1/2/L.
    Standard NLG evaluation metric for your research paper.
    """
    scorer      = rouge_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1_list, r2_list, rl_list = [], [], []
    samples     = eval_ds.select(range(min(max_samples, len(eval_ds))))

    for row in samples:
        messages_raw = row.get('messages', [])
        user_content = next(
            (m['content'] for m in messages_raw if m['role'] == 'user'), ''
        )
        reference = next(
            (m['content'] for m in messages_raw if m['role'] == 'assistant'), ''
        )

        if not user_content or not reference:
            continue

        # Build prompt with system message (same as training)
        inputs = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_content}
            ],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(device)

        input_len = inputs.shape[1]

        with torch.no_grad():
            outputs = model.generate(
                input_ids        = inputs,
                max_new_tokens   = 128,
                temperature      = 0.1,
                do_sample        = False,  # greedy for evaluation consistency
                pad_token_id     = tokenizer.eos_token_id
            )

        # Decode ONLY new tokens (fix for response parsing bug)
        prediction = tokenizer.decode(
            outputs[0][input_len:], skip_special_tokens=True
        ).strip()

        scores = scorer.score(reference, prediction)
        r1_list.append(scores['rouge1'].fmeasure)
        r2_list.append(scores['rouge2'].fmeasure)
        rl_list.append(scores['rougeL'].fmeasure)

    def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

    return {
        "ROUGE-1" : avg(r1_list),
        "ROUGE-2" : avg(r2_list),
        "ROUGE-L" : avg(rl_list),
        "n_samples": len(r1_list)
    }


print("\n📊 Computing ROUGE scores on eval set...")
rouge_scores = evaluate_rouge(model, tokenizer, eval_dataset)
print("   ROUGE scores (higher = better):")
for k, v in rouge_scores.items():
    print(f"   {k:10s} : {v}")

## 🔍 Step 12: Hallucination Probe (CUP Score)

**CUP = Consistency Under Paraphrase.** Ask the same question 3 ways using greedy decoding. If answers are inconsistent (low overlap), the model is likely hallucinating.

- CUP > 0.6 → consistent, low hallucination risk ✅  
- CUP < 0.3 → inconsistent, high hallucination risk ❌

In [ ]:
def cup_score(model, tokenizer, questions, device="cuda"):
    """
    CUP Score: Consistency Under Paraphrase.
    Novel lightweight hallucination metric that requires no ground-truth labels.
    Each question is asked in 3 phrasings. Word-overlap between answer pairs
    is measured. Average overlap = CUP score for that question.
    """
    def get_answer(question_text):
        inp = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question_text}
            ],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(device)

        input_len = inp.shape[1]

        with torch.no_grad():
            out = model.generate(
                input_ids      = inp,
                max_new_tokens = 100,
                temperature    = 0.1,
                do_sample      = False,  # Greedy — must be deterministic for CUP
                pad_token_id   = tokenizer.eos_token_id
            )
        return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

    def word_overlap(a, b):
        sa, sb = set(a.lower().split()), set(b.lower().split())
        return len(sa & sb) / max(len(sa | sb), 1)

    results    = []
    cup_scores = []

    for q in questions:
        variants = [
            q,
            f"Can you explain: {q}",
            f"I'd like to understand: {q}"
        ]
        answers = [get_answer(v) for v in variants]

        pairs  = [(0,1), (0,2), (1,2)]
        scores = [word_overlap(answers[i], answers[j]) for i, j in pairs]
        cup    = sum(scores) / 3
        cup_scores.append(cup)

        results.append({"question": q, "cup": round(cup, 3), "answers": answers})

    avg_cup = sum(cup_scores) / max(len(cup_scores), 1)
    return results, round(avg_cup, 3)


# Edit these questions to match your domain
PROBE_QUESTIONS = [
    "What is machine learning?",
    "How does gradient descent work?",
    "What is the difference between supervised and unsupervised learning?",
    "What is a neural network?",
    "How do I reset my password?"
]

print("🔍 Running CUP hallucination probe...")
probe_results, avg_cup = cup_score(model, tokenizer, PROBE_QUESTIONS)

print(f"\n📊 CUP Scores per question:")
for r in probe_results:
    flag = "✅" if r['cup'] >= 0.6 else ("⚠️" if r['cup'] >= 0.3 else "❌")
    print(f"   {flag} {r['cup']:.3f}  |  {r['question']}")

print(f"\n📊 Average CUP Score: {avg_cup}")
if avg_cup >= 0.6:
    print("   ✅ Model is CONSISTENT — low hallucination risk")
elif avg_cup >= 0.3:
    print("   ⚠️  Model is MODERATELY consistent — some hallucination risk")
else:
    print("   ❌ Model is INCONSISTENT — high hallucination risk. Consider more data or lower LR.")

## 📋 Step 13: Full Evaluation Summary

All metrics in one place for your research paper.

In [ ]:
import json

evaluation_report = {
    "model_name"              : MODEL_NAME,
    "base_model"              : "unsloth/llama-3-8b-bnb-4bit",
    "dataset"                 : DATASET_FILENAME,
    "train_examples"          : len(train_dataset),
    "eval_examples"           : len(eval_dataset),
    "perplexity_before"       : baseline_ppl,
    "perplexity_after"        : post_ppl,
    "perplexity_improvement"  : round(baseline_ppl - post_ppl, 4),
    "rouge_scores"            : rouge_scores,
    "avg_cup_score"           : avg_cup,
    "training_loss"           : float(trainer_stats.training_loss),
    "hyperparameters": {
        "learning_rate"       : LEARNING_RATE,
        "lora_rank"           : LORA_RANK,
        "lora_alpha"          : LORA_ALPHA,
        "lora_dropout"        : LORA_DROPOUT,
        "weight_decay"        : WEIGHT_DECAY,
        "scheduler"           : "cosine",
        "warmup_ratio"        : WARMUP_RATIO,
        "num_epochs"          : NUM_EPOCHS,
        "system_prompt_used"  : True
    }
}

# Save report
report_path = f"{MODEL_OUTPUT_DIR}/evaluation_report.json"
with open(report_path, "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("📋 FULL EVALUATION REPORT")
print("=" * 50)
print(json.dumps(evaluation_report, indent=2))
print("=" * 50)
print(f"\n✅ Report saved to: {report_path}")
print("\n🎉 All done! Your research-grade fine-tuned model is ready.")

## 🧪 Step 14: Quick Model Test

In [ ]:
def quick_test(user_message, max_new_tokens=128):
    """
    Quick test using correct inference approach:
    - System prompt included (matches training)
    - Only new tokens decoded (no prompt leakage)
    - repetition_penalty to reduce repetitive hallucination
    """
    inputs = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    input_len = inputs.shape[1]

    outputs = model.generate(
        input_ids          = inputs,
        max_new_tokens     = max_new_tokens,
        temperature        = 0.7,
        top_p              = 0.9,
        do_sample          = True,
        repetition_penalty = 1.15,  # Reduces repetitive hallucination loops
        pad_token_id       = tokenizer.eos_token_id
    )

    # Decode ONLY the new tokens — CRITICAL FIX (was decoding full sequence including prompt)
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()


print("🤖 Quick model test\n")
test_questions = [
    "Hello! Can you help me?",
    "What can you do for me?"
]

for q in test_questions:
    print(f"👤 User: {q}")
    print(f"🤖 Bot : {quick_test(q)}")
    print()